In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np

path = Path("mnist.npz")
if not path.exists():
    urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", path)

with np.load(path) as data:
    x = data["x_train"][:12000].reshape(-1, 784).astype("float32") / 255
    y = data["y_train"][:12000]
    test = data["x_test"][:2000].reshape(-1, 784).astype("float32") / 255
    labels = data["y_test"][:2000]

rng = np.random.default_rng(7)
w1 = rng.normal(0, 0.05, (784, 64)).astype("float32")
w2 = rng.normal(0, 0.1, (64, 10)).astype("float32")
b1, b2 = np.zeros(64, dtype="float32"), np.zeros(10, dtype="float32")
targets = np.eye(10, dtype="float32")[y]

for epoch in range(8):
    for ids in np.array_split(rng.permutation(len(x)), 120):
        batch = x[ids]
        hidden = np.maximum(batch @ w1 + b1, 0)
        logits = hidden @ w2 + b2
        probs = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs /= probs.sum(axis=1, keepdims=True)
        output_grad = (probs - targets[ids]) / len(ids)
        hidden_grad = (output_grad @ w2.T) * (hidden > 0)
        w2 -= 0.2 * (hidden.T @ output_grad)
        b2 -= 0.2 * output_grad.sum(axis=0)
        w1 -= 0.2 * (batch.T @ hidden_grad)
        b1 -= 0.2 * hidden_grad.sum(axis=0)

pred = (np.maximum(test @ w1 + b1, 0) @ w2 + b2).argmax(axis=1)
print(f"MNIST test accuracy: {(pred == labels).mean():.1%}")
print("Actual:   ", labels[:12].tolist())
print("Predicted:", pred[:12].tolist())

MNIST test accuracy: 91.4%
Actual:    [7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6]
Predicted: [7, 2, 1, 0, 4, 1, 4, 9, 6, 9, 0, 6]
